In [3]:
from cellgrn.main import normalzie_rna,parse_edges,compute_all_cells_grn,summarize_grn,format_sample_grn,format_celltype_grn
import numpy as np
import pandas as pd
import os
import anndata as ad
from scipy import sparse
import time
import gc
import pickle

In [ ]:
ps_data = np.load('/home/shaliu_fu/multireg/cellGRN_data/10X_PBMC/10X_PBMC/pseudo_data_pstime.npz',allow_pickle=True)
scRNA_data = ps_data['rna']
scATAC_data = ps_data['atac']

ps_meta = pd.DataFrame(ps_data['meta'])
cell_types = ps_meta[0]
input_gene = [i.rstrip() for i in open("/home/shaliu_fu/multireg/cellGRN_data/10X_PBMC/all_gene/10X_PBMC/input_gene.txt")]
input_peak = [i.rstrip() for i in open("/home/shaliu_fu/multireg/cellGRN_data/10X_PBMC/all_gene/10X_PBMC/input_peak.txt")]
input_tf =  [i.rstrip() for i in open("/home/shaliu_fu/multireg/cellGRN_data/10X_PBMCa/all_gene/10X_PBMC/input_tf.txt")]

ps_meta.index = [f"metacell_{i}" for i in ps_meta.index.values]
all_tf = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/db/all_hg_TF.txt")] 

In [ ]:
vars1 = pd.DataFrame(index=input_gene)
vars2 = pd.DataFrame(index=input_peak)

input_rna = ad.AnnData(X=sparse.csr_matrix(scRNA_data),obs=ps_meta,var=vars1)
input_atac = ad.AnnData(X=sparse.csr_matrix(scATAC_data),obs=ps_meta,var=vars2)


In [ ]:
soft = "linger"


outdir = f"../output/res_pbmc_metacell_{soft}/"
os.system(f"mkdir -p {outdir}")




input_df1 = pd.DataFrame(input_rna.X.toarray(),index=input_rna.obs.index.values,columns=input_rna.var.index.values)
peak_rename = [i.replace(":","-") for i in input_atac.var.index.values]
input_df2 = pd.DataFrame(input_atac.X.toarray(),index=input_atac.obs.index.values,columns=peak_rename)


cand_df = pd.read_csv(f"/home/shaliu_fu/multireg/cellGRN/data/10X_PBMC/{soft}_grn.csv",header=0)

input_genes = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN/data/10X_PBMC/{soft}_genes.txt")]
input_peaks = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN/data/10X_PBMC/{soft}_peaks.txt")]

input_peaks = [i.replace(":","-") for i in input_peaks]
input_df1 = input_df1[input_genes]
input_df2 = input_df2[input_peaks]


rna_data1,rna_data2 = normalzie_rna(input_df1)
atac_data = input_df2.copy()

input_tfs = [tf for tf in input_tf if tf in input_genes]
tf_data1 = rna_data1[input_tfs].copy()
tf_data2 = rna_data2[input_tfs].copy()

edges_idx,edges_name = parse_edges(cand_df, input_tfs, input_genes, input_peaks)

grn_scale2 = compute_all_cells_grn(tf_data2, rna_data2, atac_data,edges_idx, edges_name,
    input_tfs, input_genes, input_peaks)

with open(f"{outdir}/{soft}_cell_grn.pkl", "wb") as f:
    pickle.dump(grn_scale2, f)

sample_grn_scale2, celltype_grn_scale2 = summarize_grn(grn_scale2, cell_types)


tf_gene_res_scale2, tf_peak_res_scale2, gene_peak_res_scale2 = format_sample_grn(sample_grn_scale2)
tf_gene_ct_res_scale2, tf_peak_ct_res_scale2, gene_peak_ct_res_scale2 = format_celltype_grn(celltype_grn_scale2)




tf_gene_res_scale2.to_csv(os.path.join(outdir, "tf_gene_sample_scale2.csv"), index=False)
tf_peak_res_scale2.to_csv(os.path.join(outdir, "tf_peak_sample_scale2.csv"), index=False)
gene_peak_res_scale2.to_csv(os.path.join(outdir, "gene_peak_sample_scale2.csv"), index=False)

tf_gene_ct_res_scale2.to_csv(os.path.join(outdir, "tf_gene_celltype_scale2.csv"), index=False)
tf_peak_ct_res_scale2.to_csv(os.path.join(outdir, "tf_peak_celltype_scale2.csv"), index=False)
gene_peak_ct_res_scale2.to_csv(os.path.join(outdir, "gene_peak_celltype_scale2.csv"), index=False)